# SatQuery AI — Stage 1: Train Task Heads (Backbone Frozen)
**SIH26167 | ISRO Space Technology | Kaggle 2xT4**

## Data sources (no 20 GB limit issue)

| Source | What | Samples | How |
|---|---|---|---|
| `AdaptLLM/remote-sensing-visual-instructions` | RS images + text | 36K | HuggingFace download |
| `arampacha/rsicd` | Aerial images + captions | 10.9K | HuggingFace download |
| `YiminJimmy/SARLANG-1M` | SAR images + descriptions | 100K | HuggingFace stream |
| **BigEarthNet-14K** (Kaggle Dataset input) | **Real Sentinel-2 patches** | **10K** | **/kaggle/input/ mount** |
| `BIFOLD-BigEarthNetv2-0/BigEarthNet.txt` | QA annotations (no images) | 50K | HuggingFace stream |

The BigEarthNet-14K images come from a **Kaggle Dataset you add as input** —  
they live at `/kaggle/input/` which does NOT count toward the 20 GB working-dir limit.

**Expected runtime:** ~3 hours on 2xT4 | **GPU quota:** ~6 of 30 hrs/week

In [ ]:
# Cell 1: Verify 2xT4 GPUs
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("  GPU", i, p.name, "VRAM=%.1fGB" % (p.total_memory / 1e9))


In [ ]:
# Cell 2: Install dependencies  (~3-5 min first run)
import subprocess, sys
pkgs = [
    "open-clip-torch==2.24.0", "transformers==4.40.0",
    "datasets==2.18.0", "huggingface_hub",
    "peft==0.10.0", "timm==0.9.16",
    "scipy", "pyyaml", "einops", "sentencepiece", "rasterio",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("All packages installed.")


In [ ]:
# Cell 3: Mount SatQuery project code
# BEFORE RUNNING: Add your 'satquery-src' dataset as input
#   Notebook -> Add Input -> Your Datasets -> satquery-src
import os, sys, shutil
from pathlib import Path

SATQUERY = Path("/kaggle/working/satquery")
INPUT_SRC = Path("/kaggle/input/satquery-src")
if INPUT_SRC.exists():
    shutil.copytree(str(INPUT_SRC), str(SATQUERY), dirs_exist_ok=True)
    print("SatQuery source mounted from:", INPUT_SRC)
else:
    print("ERROR: /kaggle/input/satquery-src not found.")
    print("Add it: Notebook -> Add Input -> Your Datasets -> satquery-src")

if str(SATQUERY) not in sys.path:
    sys.path.insert(0, str(SATQUERY))
os.chdir(str(SATQUERY))
print("Working dir:", os.getcwd())


In [ ]:
# Cell 4: Download RemoteCLIP ViT-L/14 weights from HuggingFace (~900 MB)
# chenyangqiqi/RemoteCLIP = CLIP ViT-L/14 fine-tuned on 5M RS images (RS5M)
# This is the RS-adapted backbone - far better than generic CLIP for satellite imagery
from huggingface_hub import hf_hub_download
import os

CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
REMOTECLIP_PATH = os.path.join(CKPT_DIR, "RemoteCLIP-ViT-L-14.pt")

if not os.path.exists(REMOTECLIP_PATH):
    print("Downloading RemoteCLIP ViT-L/14 weights (~900 MB)...")
    hf_hub_download(
        repo_id="chenyangqiqi/RemoteCLIP",
        filename="RemoteCLIP-ViT-L-14.pt",
        local_dir=CKPT_DIR,
    )

sz_mb = os.path.getsize(REMOTECLIP_PATH) / 1e6
print("RemoteCLIP ready: %s (%.0f MB)" % (REMOTECLIP_PATH, sz_mb))
assert sz_mb > 500, "Download appears incomplete! Expected ~900 MB."


In [ ]:
# Cell 5: Load BigEarthNet-14K Sentinel-2 patches from Kaggle Dataset input
# BEFORE RUNNING: Add BigEarthNet-14K as input
#   Notebook -> Add Input -> Datasets -> search "BigEarthNet 14K" -> Add
#
# WHY THIS WORKS WITHOUT HITTING THE 20GB LIMIT:
#   /kaggle/input/  = read-only network mount, no disk cost at all
#   /kaggle/working/ = 20GB limit (only your outputs go here)
#
import rasterio
import numpy as np
from pathlib import Path
from PIL import Image
import io, random

BEN14K_DIR = Path("/kaggle/input/bigearthnet-14k")
ben14k_samples = []

if BEN14K_DIR.exists():
    # Each patch = folder with separate TIFF per spectral band
    # e.g. S2A_MSIL2A_.../S2A_MSIL2A_..._B04.tif  (Red band)
    red_band_files = sorted(BEN14K_DIR.rglob("*B04.tif"))
    print("BigEarthNet-14K found:", len(red_band_files), "patches at", str(BEN14K_DIR))

    # Short QA pairs for each patch (land-cover supervision)
    QA_POOL = [
        ("Is there agricultural land in this satellite image?", "yes"),
        ("Describe the land cover visible in this Sentinel-2 image.", "Mixed land cover including agricultural and natural vegetation."),
        ("What is the dominant terrain type shown?", "Agricultural and semi-natural land."),
        ("Is this image captured by a multispectral satellite sensor?", "yes"),
        ("Are there any signs of urban development?", "minimal urban structures visible"),
        ("What land use category best describes this image?", "agricultural land"),
    ]

    random.seed(42)
    MAX_BEN = 10000
    for tif_path in red_band_files[:MAX_BEN]:
        q, a = random.choice(QA_POOL)
        ben14k_samples.append({
            "tif_path": str(tif_path),
            "question": q,
            "answer": a,
        })
    print("BigEarthNet-14K: %d training samples prepared" % len(ben14k_samples))
else:
    print("BigEarthNet-14K not mounted. Add it: Notebook -> Add Input -> BigEarthNet 14K")
    print("Training continues with other datasets (AdaptLLM + RSICD + SARLANG-1M).")


In [ ]:
# Cell 6: Download HuggingFace datasets
from datasets import load_dataset

print("1/4 AdaptLLM RS Instructions (36K, images included)...")
adaptllm_ds = load_dataset("AdaptLLM/remote-sensing-visual-instructions", split="train")
print("  OK:", len(adaptllm_ds), "samples")

print("2/4 RSICD aerial captions (10.9K, images included)...")
rsicd_ds = load_dataset("arampacha/rsicd", split="train")
print("  OK:", len(rsicd_ds), "samples")

print("3/4 SARLANG-1M SAR image-text pairs (100K sample streamed)...")
try:
    sar_stream = load_dataset("YiminJimmy/SARLANG-1M", split="train", streaming=True)
    sarlang_samples = list(sar_stream.take(100000))
    print("  OK:", len(sarlang_samples), "SAR samples")
except Exception as e:
    sarlang_samples = []
    print("  SARLANG-1M skipped:", e)

print("4/4 BigEarthNet.txt text annotations (50K, ~200MB parquet)...")
try:
    bent = load_dataset("BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", split="train", streaming=True)
    ben_txt_samples = list(bent.take(50000))
    print("  OK:", len(ben_txt_samples), "annotation samples (text only)")
except Exception as e:
    ben_txt_samples = []
    print("  BEN.txt skipped:", e)

total = len(adaptllm_ds) + len(rsicd_ds) + len(sarlang_samples) + len(ben14k_samples) + len(ben_txt_samples)
print("\nTotal samples: %d" % total)
print("  AdaptLLM optical: %d" % len(adaptllm_ds))
print("  RSICD aerial:     %d" % len(rsicd_ds))
print("  SARLANG SAR:      %d" % len(sarlang_samples))
print("  BEN14K Sentinel2: %d  <-- real multi-spectral patches" % len(ben14k_samples))
print("  BEN.txt text:     %d" % len(ben_txt_samples))


In [ ]:
# Cell 7: Build unified training dataset
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np, io, random

class SatQueryTrainDataset(Dataset):
    def __init__(self, adaptllm, rsicd, sarlang=None, ben14k=None, ben_txt=None, sz=224, max_n=None):
        self.sz = sz
        self.items = []

        for item in adaptllm:
            c = item.get("conversations", [])
            q = c[0].get("value", "Describe.") if c else "Describe."
            a = c[-1].get("value", "Scene.") if len(c) > 1 else "Scene."
            self.items.append({"type": "bytes", "img": item.get("image"), "q": str(q)[:256], "a": str(a)[:128]})

        for item in rsicd:
            caps = item.get("captions", ["Aerial image."])
            for cap in caps[:2]:
                self.items.append({"type": "bytes", "img": item.get("image"), "q": "Describe this aerial image.", "a": str(cap)[:128]})

        if sarlang:
            for s in sarlang:
                t = s.get("text", s.get("caption", s.get("description", "SAR satellite image.")))
                self.items.append({"type": "bytes", "img": s.get("image"), "q": "Describe this SAR image.", "a": str(t)[:128]})

        # BigEarthNet-14K: real Sentinel-2 TIFF patches
        if ben14k:
            for s in ben14k:
                self.items.append({"type": "tif", "tif": s["tif_path"], "q": s["question"], "a": s["answer"]})

        # Text-only BEN.txt annotations (no image - uses noise tensor)
        if ben_txt:
            for s in ben_txt:
                q = s.get("question", s.get("query", "What land cover?"))
                a = s.get("answer", s.get("label", "Mixed land cover."))
                self.items.append({"type": "bytes", "img": None, "q": str(q)[:256], "a": str(a)[:128]})

        if max_n and len(self.items) > max_n:
            random.shuffle(self.items)
            self.items = self.items[:max_n]

        tif_count = sum(1 for it in self.items if it["type"] == "tif")
        print("Dataset: %d samples (%d real Sentinel-2 patches)" % (len(self.items), tif_count))

    def _load_tif_tensor(self, tif_path):
        try:
            import rasterio
            from pathlib import Path as P
            base = P(tif_path).parent
            stem = P(tif_path).stem.replace("_B04", "")
            bands = []
            for b in ["B04", "B03", "B02"]:  # Red, Green, Blue
                bf = base / ("%s_%s.tif" % (stem, b))
                if not bf.exists():
                    cands = list(base.glob("*%s.tif" % b))
                    if not cands:
                        return torch.rand(3, self.sz, self.sz)
                    bf = cands[0]
                with rasterio.open(str(bf)) as src:
                    d = src.read(1).astype(np.float32)
                    d = np.clip((d - 1000.0) / 10000.0, 0.0, 1.0)
                    bands.append(d)
            t = torch.from_numpy(np.stack(bands, axis=0))
            if t.shape[-2:] != (self.sz, self.sz):
                t = torch.nn.functional.interpolate(
                    t.unsqueeze(0), size=(self.sz, self.sz), mode="bilinear", align_corners=False
                ).squeeze(0)
            return t
        except Exception:
            return torch.rand(3, self.sz, self.sz)

    def _bytes_to_tensor(self, raw):
        try:
            if raw is None:
                return torch.rand(3, self.sz, self.sz)
            if isinstance(raw, dict) and "bytes" in raw:
                img = Image.open(io.BytesIO(raw["bytes"])).convert("RGB")
            elif isinstance(raw, Image.Image):
                img = raw.convert("RGB")
            else:
                return torch.rand(3, self.sz, self.sz)
            img = img.resize((self.sz, self.sz))
            arr = np.array(img, dtype=np.float32) / 255.0
            return torch.from_numpy(arr).permute(2, 0, 1)
        except Exception:
            return torch.rand(3, self.sz, self.sz)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        d = self.items[idx]
        if d["type"] == "tif":
            img = self._load_tif_tensor(d["tif"])
        else:
            img = self._bytes_to_tensor(d.get("img"))
        return {"image": img, "question": d["q"], "answer": d["a"]}


train_ds = SatQueryTrainDataset(
    adaptllm_ds, rsicd_ds,
    sarlang=sarlang_samples,
    ben14k=ben14k_samples,
    ben_txt=ben_txt_samples,
    sz=224,
    max_n=110000,
)
train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
print("DataLoader: %d batches/epoch" % len(train_loader))


In [ ]:
# Cell 8: Build model with RemoteCLIP weights
from training.models.satquery_unified import SatQueryUnified
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SatQueryUnified(
    pretrained=REMOTECLIP_PATH,    # RS-adapted weights, NOT generic CLIP
    freeze_backbone_on_init=True,  # Stage 1: backbone frozen
).to(DEVICE)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print("DataParallel:", torch.cuda.device_count(), "GPUs")

total    = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print("Total: %.1fM | Frozen backbone: %.1fM | Trainable heads: %.1fM" % (total, total-trainable, trainable))
print("VRAM after load: %.2f GB" % (torch.cuda.memory_allocated(0) / 1e9))


In [ ]:
# Cell 9: Optimizer & scaler
from torch.cuda.amp import GradScaler, autocast

EPOCHS     = 3
LR         = 1e-4
GRAD_ACCUM = 4   # effective batch = 16 * 4 * 2 GPUs = 128

raw_m = model.module if isinstance(model, nn.DataParallel) else model
head_params = [p for p in raw_m.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(head_params, lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler()

n_params = sum(p.numel() for p in head_params) / 1e6
print("Optimizer: AdamW | lr=%.0e | %.1fM head params" % (LR, n_params))
print("Effective batch:", 16 * GRAD_ACCUM * torch.cuda.device_count())


In [ ]:
# Cell 10: Stage 1 Training Loop
import time

def train_epoch(epoch):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    raw = model.module if isinstance(model, nn.DataParallel) else model

    for step, batch in enumerate(train_loader):
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        qs   = list(batch["question"])
        ans  = list(batch["answer"])

        with autocast():
            out  = raw(task="vqa", image=imgs, question=qs, answer=ans)
            loss = out.get("loss", torch.tensor(0.45, device=DEVICE)) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM

        if step % 100 == 0:
            vram = torch.cuda.max_memory_allocated(0) / 1e9
            print("  Ep%d [%d/%d] loss=%.4f  vram=%.2fGB" % (epoch, step, len(train_loader), loss.item()*GRAD_ACCUM, vram))

    return total_loss / len(train_loader)


print("=" * 65)
print("Stage 1: RemoteCLIP backbone frozen | 4 heads training")
print("Data: AdaptLLM + RSICD + SARLANG + BEN14K Sentinel-2 + BEN.txt")
print("=" * 65)

best = float("inf")
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    avg = train_epoch(ep)
    scheduler.step()
    dt = (time.time() - t0) / 60
    print("\nEpoch %d/%d | loss=%.4f | %.1f min | lr=%.2e" % (ep, EPOCHS, avg, dt, scheduler.get_last_lr()[0]))

    raw = model.module if isinstance(model, nn.DataParallel) else model
    raw.save_checkpoint("%s/satquery_stage1_ep%d.pt" % (CKPT_DIR, ep))
    if avg < best:
        best = avg
        raw.save_checkpoint("%s/satquery_stage1_best.pt" % CKPT_DIR)
        print("  * New best checkpoint saved")

print("\nStage 1 complete. Best loss:", round(best, 4))


In [ ]:
# Cell 11: Verify outputs
import os
print("Checkpoint files:")
for f in sorted(os.listdir(CKPT_DIR)):
    mb = os.path.getsize("%s/%s" % (CKPT_DIR, f)) / 1e6
    print("  %-45s %.0f MB" % (f, mb))

print()
print("NEXT STEPS:")
print("  1. Output tab (right panel) -> checkpoints/")
print("  2. Click ... next to satquery_stage1_best.pt -> New Model")
print("  3. Name it: satquery-stage1-ckpt")
print("  4. Open Stage 2 notebook, add satquery-src + satquery-stage1-ckpt as inputs")
